# Worked Capstone: Demand Forecasting with Time-Aware Validation

**Domain:** Time-series supervised learning  
**Primary dataset:** `demand_timeseries.csv`  
**Level:** Practitioner to Advanced

## Business goal

Forecast daily demand using only information available before prediction time and compare against seasonal naive baselines.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What forecast horizon is assumed?
2. Which lag features are available?
3. Does ML beat a seasonal baseline?
4. How stable is error over time?

        ## Definition of done

        - [ ] Temporal features
- [ ] Leakage-safe lags
- [ ] Seasonal baseline
- [ ] ML model
- [ ] 2025 holdout
- [ ] Monthly error slices
- [ ] Saved model

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Rolling features include current/future demand | Shift before rolling. |
| Random split inflates performance | Use chronological evaluation. |
| Promotions unknown at forecast time | Include only scheduled information. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Create leakage-safe features

All target-derived features are shifted so row t never uses demand at t or later.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
import joblib

df=pd.read_csv(DATA_DIR/"demand_timeseries.csv",parse_dates=["date"]).sort_values("date")
df["day_of_week"]=df.date.dt.dayofweek
df["month"]=df.date.dt.month
df["day_of_year"]=df.date.dt.dayofyear
df["trend"]=np.arange(len(df))
for lag in [1,7,14,28]:
    df[f"lag_{lag}"]=df.demand.shift(lag)
df["rolling_7_mean"]=df.demand.shift(1).rolling(7).mean()
df["rolling_28_mean"]=df.demand.shift(1).rolling(28).mean()
model_data=df.dropna().copy()
features=["promotion","holiday","day_of_week","month","day_of_year","trend",
          "lag_1","lag_7","lag_14","lag_28","rolling_7_mean","rolling_28_mean"]
train=model_data[model_data.date<"2025-01-01"]
test=model_data[model_data.date>="2025-01-01"]
print(train.date.min(),train.date.max(),test.date.min(),test.date.max())

## 2. Baselines

Use yesterday and the same weekday last week as transparent baselines.

In [ ]:
baseline_rows=[]
for name,pred in {"lag_1":test.lag_1,"seasonal_lag_7":test.lag_7}.items():
    baseline_rows.append({
        "model":name,
        "MAE":mean_absolute_error(test.demand,pred),
        "RMSE":mean_squared_error(test.demand,pred)**.5,
    })
display(pd.DataFrame(baseline_rows))

## 3. Fit model and evaluate

Tune only on past data in a fuller project; here we use a fixed, regularized histogram gradient booster.

In [ ]:
model=HistGradientBoostingRegressor(
    learning_rate=.05,max_iter=240,max_leaf_nodes=18,l2_regularization=2.0,random_state=42
)
model.fit(train[features],train.demand)
test["prediction"]=model.predict(test[features])
test["absolute_error"]=(test.demand-test.prediction).abs()
result={
    "model":"HistGradientBoostingRegressor",
    "MAE":mean_absolute_error(test.demand,test.prediction),
    "RMSE":mean_squared_error(test.demand,test.prediction)**.5,
}
print(result)

## 4. Error over time

Aggregate errors by month and inspect forecast traces.

In [ ]:
monthly_error=test.set_index("date").resample("MS").absolute_error.agg(["mean","median","max"])
display(monthly_error.round(2))
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(test.date,test.demand,label="Actual",linewidth=1)
ax.plot(test.date,test.prediction,label="Prediction",linewidth=1)
ax.set(title="2025 daily demand forecast",xlabel="Date",ylabel="Demand")
ax.legend(); plt.show()

## 5. Save artifact and scope

Store feature names and horizon assumptions beside the model.

In [ ]:
artifact=ARTIFACT_DIR/"capstone_demand_model.joblib"
joblib.dump(model,artifact)
metadata={
    "features":features,
    "evaluation_period":[str(test.date.min().date()),str(test.date.max().date())],
    "assumed_horizon":"one day ahead",
    "known_future_features":["promotion","holiday"],
    "metrics":{k:(float(v) if isinstance(v,(float,np.floating)) else v) for k,v in result.items()},
}
(ARTIFACT_DIR/"capstone_demand_metadata.json").write_text(json.dumps(metadata,indent=2))
print(metadata)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.